In [1]:
import pandas as pd
import numpy as np
import joblib

ROOT = r"C:\Users\Acer\Documents\AuthentiScore"

FEATURES = [
    "followers",
    "following",
    "posts",
    "bio_length",
    "has_profile_picture",
    "is_private",
    "username_digit_ratio",
    "follower_following_ratio",
    "profile_completeness"
]

print("M17 Unified Risk Engine")

M17 Unified Risk Engine


In [2]:
model = joblib.load(
    ROOT + r"\backend\trained_models\authentiscore_rf_v1.pkl"
)

feature_list = joblib.load(
    ROOT + r"\backend\trained_models\authentiscore_features_v1.pkl"
)

print("Model loaded successfully!")
print("Features:", feature_list)

Model loaded successfully!
Features: ['followers', 'following', 'posts', 'bio_length', 'has_profile_picture', 'is_private', 'username_digit_ratio', 'follower_following_ratio', 'profile_completeness']


In [3]:
df = pd.read_csv(
    ROOT + r"\data\processed\instagram_features.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (696, 10)


In [4]:
def calculate_ml_risk(profile):

    X = pd.DataFrame(
        [[profile[feature] for feature in feature_list]],
        columns=feature_list
    )

    probability = model.predict_proba(X)[0][1]

    return round(float(probability * 100), 2)

In [5]:
ML_WEIGHT = 0.60
NLP_WEIGHT = 0.25
RULE_WEIGHT = 0.15


def calculate_unified_risk(
    ml_risk,
    nlp_risk,
    rule_risk
):

    final_score = (
        ml_risk * ML_WEIGHT
        + nlp_risk * NLP_WEIGHT
        + rule_risk * RULE_WEIGHT
    )

    return round(min(max(final_score, 0), 100), 2)

In [6]:
def classify_risk(score):

    if score < 30:
        return "Low Risk"

    elif score < 60:
        return "Medium Risk"

    elif score < 80:
        return "High Risk"

    else:
        return "Very High Risk"

In [8]:
def calculate_profile_rule_risk(profile):

    risk_score = 0
    signals = []

    # 1. Low follower/following ratio
    if profile["following"] > 0:

        ratio = profile["followers"] / profile["following"]

        if ratio < 0.1:
            risk_score += 20

            signals.append({
                "rule": "low_follower_following_ratio",
                "weight": 20,
                "message": "Very low follower-to-following ratio"
            })

    # 2. High username digit ratio
    if profile["username_digit_ratio"] > 0.5:

        risk_score += 15

        signals.append({
            "rule": "high_username_digit_ratio",
            "weight": 15,
            "message": "Username contains a high proportion of digits"
        })

    # 3. Missing profile picture
    if profile["has_profile_picture"] == 0:

        risk_score += 20

        signals.append({
            "rule": "missing_profile_picture",
            "weight": 20,
            "message": "Profile picture is missing"
        })

    # 4. Low profile completeness
    if profile["profile_completeness"] < 0.6:

        risk_score += 15

        signals.append({
            "rule": "low_profile_completeness",
            "weight": 15,
            "message": "Profile has low completeness"
        })

    # 5. No posts
    if profile["posts"] == 0:

        risk_score += 10

        signals.append({
            "rule": "no_posts",
            "weight": 10,
            "message": "Profile has no posts"
        })

    # 6. Empty bio
    if profile["bio_length"] == 0:

        risk_score += 10

        signals.append({
            "rule": "empty_bio",
            "weight": 10,
            "message": "Profile has no biography"
        })

    risk_score = min(risk_score, 100)

    return {
        "profile_rule_risk": risk_score,
        "signals": signals
    }

In [9]:
profile = df.iloc[0]

ml_risk = calculate_ml_risk(profile)

# Temporary NLP score for testing
nlp_risk = 0

# Calculate profile rule risk
rule_result = calculate_profile_rule_risk(profile)

rule_risk = rule_result["profile_rule_risk"]

final_score = calculate_unified_risk(
    ml_risk,
    nlp_risk,
    rule_risk
)

classification = classify_risk(final_score)

print("=" * 60)
print("AUTHENTISCORE RESULT")
print("=" * 60)

print("ML Risk:", ml_risk)
print("NLP Risk:", nlp_risk)
print("Rule Risk:", rule_risk)
print("Final Risk Score:", final_score)
print("Risk Classification:", classification)

print("\nRule Signals:")
for signal in rule_result["signals"]:
    print(signal)

AUTHENTISCORE RESULT
ML Risk: 3.67
NLP Risk: 0
Rule Risk: 0
Final Risk Score: 2.2
Risk Classification: Low Risk

Rule Signals:
